# CU28 mixed_context - Synthetic Plant Layer EDA

Notebook narrativo de auditoria para el scope `mixed_context`.


## Objetivo

Validar de forma plausible la capa operativa sintetica que aproxima inventario, requirement, lead time, yield y waste para la ruta mixed_context.


## Alcance

Este analisis describe la ruta oficial reproducible `mixed_context`. Las senales externas se tratan como contexto/proxy. Las variables internas de planta siguen siendo sinteticas salvo carga posterior de cliente.


## Inputs

            - `data/processed/synthetic/plant/synthetic_plant_layer__mixed_context.csv`
- `data/processed/synthetic/plant/synthetic_plant_metadata__mixed_context.json`
- `config/manufacturing_profiles.yaml`
- `docs/simulation_assumptions.md`
- `docs/simulation_data_basis.md`


## Outputs esperados

            - `reports/tables/eda/synthetic_layer_summary__mixed_context.csv`
- `reports/tables/eda/synthetic_layer_by_profile__mixed_context.csv`
- `reports/tables/eda/synthetic_layer_variable_ranges__mixed_context.csv`
- `reports/figures/eda/synthetic_requirement_by_profile__mixed_context.png`
- `reports/figures/eda/synthetic_inventory_distribution__mixed_context.png`
- `reports/figures/eda/synthetic_lead_time_by_profile__mixed_context.png`
- `reports/figures/eda/synthetic_yield_waste_by_profile__mixed_context.png`
- `reports/figures/eda/synthetic_procurement_need_by_profile__mixed_context.png`
- `reports/figures/eda/synthetic_layer_correlation__mixed_context.png`
- `reports/figures/eda/synthetic_requirement_inventory_timeseries__mixed_context.png`


## Limitaciones

Este notebook documenta evidencia reproducible del pipeline oficial, pero no sustituye la revision de codigo, la auditoria de datos de origen ni una certificacion operacional de planta.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from src.reproducibility.notebook_support import (
    detect_temporal_columns,
    ensure_eda_dirs,
    execution_metadata,
    first_valid_temporal_range,
    load_source_manifests,
    load_tabular_file,
    parse_markdown_table,
    print_frame,
    print_series,
    project_root,
    read_json,
    relative_to_root,
    save_figure,
    save_table,
    sha256_file,
)

import yaml


In [ ]:
NOTEBOOK_NAME = "03_synthetic_plant_layer_eda.ipynb"
PROJECT_ROOT = project_root()
SCOPE = globals().get("scope", "mixed_context")
REPORT_DIRS = ensure_eda_dirs()
META = execution_metadata(SCOPE)
FIGURES = []
TABLES = []
print(json.dumps(META, indent=2))


## Carga de datos

Las siguientes celdas cargan los artefactos de entrada y muestran verificaciones intermedias antes de producir tablas y graficas.


In [ ]:
synthetic_path = PROJECT_ROOT / "data/processed/synthetic/plant/synthetic_plant_layer__mixed_context.csv"
synthetic_meta_path = PROJECT_ROOT / "data/processed/synthetic/plant/synthetic_plant_metadata__mixed_context.json"
profiles_path = PROJECT_ROOT / "config" / "manufacturing_profiles.yaml"
synthetic_df = pd.read_csv(synthetic_path)
synthetic_df["date"] = pd.to_datetime(synthetic_df["date"], errors="coerce")
synthetic_meta = read_json(synthetic_meta_path)
print(synthetic_df.shape)
print_frame("Synthetic layer preview", synthetic_df.head(10))


In [ ]:
manufacturing_profiles = yaml.safe_load(profiles_path.read_text(encoding="utf-8"))
selected_profile_keys = list((manufacturing_profiles or {}).keys())
print({"profile_count": len(selected_profile_keys), "profiles": selected_profile_keys[:10]})
print(json.dumps({k: synthetic_meta[k] for k in ["environment_name", "time_granularity", "validated_base_horizon_label"] if k in synthetic_meta}, indent=2))


## Inspeccion inicial

Se revisan shape, columnas, tipos y nulos para dejar claro que esta capa es sintetica y que su funcion es dar contexto operativo reproducible al pipeline mixto.


In [ ]:
shape_summary = pd.DataFrame([{"rows": len(synthetic_df), "columns": len(synthetic_df.columns)}])
column_summary = pd.DataFrame({"column": synthetic_df.columns})
print_frame("Shape summary", shape_summary)
print_frame("Column list", column_summary, rows=30)


In [ ]:
dtype_summary = synthetic_df.dtypes.astype(str).reset_index()
dtype_summary.columns = ["column", "dtype"]
print_frame("Dtype summary", dtype_summary, rows=30)
display(dtype_summary.head(30))


In [ ]:
null_summary = synthetic_df.isna().mean().reset_index()
null_summary.columns = ["column", "missing_pct"]
null_summary["missing_pct"] = null_summary["missing_pct"].round(4)
print_frame("Null summary", null_summary.sort_values("missing_pct", ascending=False), rows=30)
display(null_summary.sort_values("missing_pct", ascending=False).head(30))


In [ ]:
variables_of_interest = [
    "current_inventory_tons",
    "expected_requirement_tons",
    "lead_time_days",
    "safety_coverage_days",
    "expected_yield_rate",
    "expected_waste_rate",
    "synthetic_procurement_need",
]
variable_ranges = synthetic_df[variables_of_interest].agg(["min", "max", "mean"]).transpose().reset_index()
variable_ranges.columns = ["variable", "min", "max", "mean"]
print_frame("Observed ranges", variable_ranges, rows=20)
display(variable_ranges)


## Analisis por variable

Las siguientes celdas revisan la plausibilidad operativa de requirement, inventory, lead time, coverage, yield, waste y la senal upstream sintetica.


In [ ]:
requirement_summary = synthetic_df.groupby("destination_profile")["expected_requirement_tons"].agg(["count", "mean", "median", "min", "max"]).reset_index()
print_frame("Requirement by destination_profile", requirement_summary, rows=20)
display(requirement_summary)


In [ ]:
inventory_summary = synthetic_df.groupby("destination_profile")["current_inventory_tons"].agg(["mean", "median", "min", "max"]).reset_index()
print_frame("Inventory by destination_profile", inventory_summary, rows=20)
display(inventory_summary)


In [ ]:
lead_coverage_summary = synthetic_df.groupby("destination_profile")[["lead_time_days", "safety_coverage_days"]].mean().reset_index()
print_frame("Lead time and safety coverage by profile", lead_coverage_summary, rows=20)
display(lead_coverage_summary)


In [ ]:
yield_waste_summary = synthetic_df.groupby("destination_profile")[["expected_yield_rate", "expected_waste_rate"]].mean().reset_index()
print_frame("Yield and waste by profile", yield_waste_summary, rows=20)
display(yield_waste_summary)


In [ ]:
profile_summary = synthetic_df.groupby("destination_profile").agg(
    rows=("destination_profile", "size"),
    requirement_mean=("expected_requirement_tons", "mean"),
    inventory_mean=("current_inventory_tons", "mean"),
    procurement_need_mean=("synthetic_procurement_need", "mean"),
).reset_index()
print_frame("Overall summary by profile", profile_summary, rows=20)
display(profile_summary)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(requirement_summary["destination_profile"], requirement_summary["mean"], color="#355c7d")
ax.set_title("Expected requirement by destination profile")
ax.set_ylabel("tons")
FIGURES.append(save_figure(fig, "synthetic_requirement_by_profile__mixed_context.png"))
plt.close(fig)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(synthetic_df["current_inventory_tons"], bins=20, color="#6c5b7b", edgecolor="white")
ax.set_title("Current inventory distribution")
ax.set_xlabel("tons")
FIGURES.append(save_figure(fig, "synthetic_inventory_distribution__mixed_context.png"))
plt.close(fig)


In [ ]:
lead_time_plot = synthetic_df.groupby("destination_profile")["lead_time_days"].mean().reset_index()
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(lead_time_plot["destination_profile"], lead_time_plot["lead_time_days"], color="#c06c84")
ax.set_title("Lead time by destination profile")
ax.set_ylabel("days")
FIGURES.append(save_figure(fig, "synthetic_lead_time_by_profile__mixed_context.png"))
plt.close(fig)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
width = 0.35
positions = np.arange(len(yield_waste_summary))
ax.bar(positions - width / 2, yield_waste_summary["expected_yield_rate"], width=width, label="yield_rate")
ax.bar(positions + width / 2, yield_waste_summary["expected_waste_rate"], width=width, label="waste_rate")
ax.set_xticks(positions)
ax.set_xticklabels(yield_waste_summary["destination_profile"], rotation=15)
ax.set_title("Yield and waste by destination profile")
ax.legend()
FIGURES.append(save_figure(fig, "synthetic_yield_waste_by_profile__mixed_context.png"))
plt.close(fig)


In [ ]:
procurement_need_summary = synthetic_df.groupby("destination_profile")["synthetic_procurement_need"].mean().reset_index()
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(procurement_need_summary["destination_profile"], procurement_need_summary["synthetic_procurement_need"], color="#2a9d8f")
ax.set_title("Synthetic procurement need by profile")
ax.set_ylabel("proxy tons")
FIGURES.append(save_figure(fig, "synthetic_procurement_need_by_profile__mixed_context.png"))
plt.close(fig)
print_frame("Synthetic procurement need by profile", procurement_need_summary, rows=20)


In [ ]:
operational_cols = [
    "current_inventory_tons",
    "expected_requirement_tons",
    "lead_time_days",
    "safety_coverage_days",
    "expected_yield_rate",
    "expected_waste_rate",
    "synthetic_procurement_need",
]
corr = synthetic_df[operational_cols].corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(7, 6))
image = ax.imshow(corr.values, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.index)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticklabels(corr.index)
ax.set_title("Operational variable correlation")
fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
FIGURES.append(save_figure(fig, "synthetic_layer_correlation__mixed_context.png"))
plt.close(fig)


In [ ]:
weekly_profile = synthetic_df.groupby("date")[["expected_requirement_tons", "current_inventory_tons"]].mean().reset_index()
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(weekly_profile["date"], weekly_profile["expected_requirement_tons"], label="expected_requirement_tons")
ax.plot(weekly_profile["date"], weekly_profile["current_inventory_tons"], label="current_inventory_tons")
ax.set_title("Weekly requirement and inventory")
ax.set_ylabel("tons")
ax.legend()
FIGURES.append(save_figure(fig, "synthetic_requirement_inventory_timeseries__mixed_context.png"))
plt.close(fig)


## Interpretacion

`destination_profile` representa destino productivo previsto y no una compra observada. `synthetic_procurement_need` es una senal upstream de presion y no la cantidad final recomendada. La decision final de cantidad aparece despues como `order_quantity_tons`.


In [ ]:
TABLES.append(save_table(variable_ranges, "synthetic_layer_variable_ranges__mixed_context.csv"))
TABLES.append(save_table(profile_summary, "synthetic_layer_by_profile__mixed_context.csv"))
TABLES.append(save_table(null_summary, "synthetic_layer_summary__mixed_context.csv"))
RESULT = {
    "notebook": NOTEBOOK_NAME,
    "tables": TABLES,
    "figures": FIGURES,
    "findings": [
        "The synthetic layer produces plausible operational ranges by destination profile.",
        "The upstream signal synthetic_procurement_need behaves as a pressure indicator and not as a final order quantity.",
        "Inventory, requirement, lead time, yield and waste remain synthetic unless the customer uploads observed values.",
    ],
    "limitations": [
        "This notebook validates plausibility only; it does not convert synthetic variables into observed plant evidence.",
    ],
}
print(json.dumps(RESULT, indent=2))


## Limitaciones

La capa sintetica es una decision metodologica declarada del caso de uso. Sirve para reproducibilidad y simulacion controlada, no para afirmar observacion directa de operaciones internas.


## Concluson final

La evidencia visual confirma que la capa sintetica es coherente con el relato oficial: soporte batch/offline, datos externos de contexto y variables operativas sinteticas salvo carga futura de cliente.
